# Installing Dependencies


In [1]:
%pip install lightgbm xgboost scikit-learn numpy pandas


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

# Template required variables
TRAIN_LABEL = train_labels
TEST_DATA   = test_labels

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

# ACC magnitude
for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

# EEG log1p (must happen before any processing)
EEG_COLS = ['delta','theta','lowAlpha','highAlpha','lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)
print('Train PIDs:', sorted(train_labels.pid.unique()))
print('Test  PIDs:', sorted(test_labels.pid.unique()))

Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)
Train PIDs: ['01Z2', '70N8', '7PF3', 'CQ2G', 'D1XP', 'DT5C', 'F1ZM', 'LIUY', 'SE4Q', 'TPQI', 'Y21H']
Test  PIDs: ['13P2', '2XO3', '43JW', 'C8Q6', 'HDS9', 'NQRB', 'OL6N', 'P4DZ', 'QEYR', 'SNG7', 'TF0Y', 'WZDL']


## 1. Per-Subject Session Baseline Statistics

In [3]:
def compute_subject_baseline(sensor_df, val_col, pid_col='pid'):
    """Compute per-subject mean and std from the entire session.
    Returns dict: pid -> (mean, std)
    """
    baseline = {}
    for pid, grp in sensor_df.groupby(pid_col):
        vals = grp[val_col].dropna()
        baseline[pid] = (vals.mean(), vals.std() + 1e-8)
    return baseline


# Pre-compute baselines for all sensors (train + test combined for test inference)
# For test subjects, we only have their own test data — so compute separately
bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')

print('Baselines computed.')
print('HR baseline (train) — sample:')
for pid, (m, s) in list(bl_tr_hr.items())[:3]:
    print(f'  {pid}: mean={m:.1f} bpm, std={s:.1f}')

Baselines computed.
HR baseline (train) — sample:
  01Z2: mean=84.2 bpm, std=4.3
  70N8: mean=84.5 bpm, std=11.0
  7PF3: mean=82.6 bpm, std=10.6


## 2. Multi-Window Feature Extraction

In [4]:
WINDOWS_MS = [2500, 5000, 10000]  # ±2.5s, ±5s, ±10s


def extract_window_stats(vals, win_label, sensor_name):
    """Given a numpy array of values in a window, return feature dict."""
    feat = {}
    prefix = f'{sensor_name}_{win_label}'
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_min']   = np.min(vals)
        feat[f'{prefix}_max']   = np.max(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
        feat[f'{prefix}_iqr']   = np.percentile(vals, 75) - np.percentile(vals, 25)
    else:
        for suf in ['mean','std','min','max','range','slope','p25','p75','iqr']:
            feat[f'{prefix}_{suf}'] = np.nan
    return feat


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc):
    """
    For each label row, extract features from multiple window sizes.
    Includes: raw stats, per-subject deviation (z-score relative to session baseline),
              EDA artifact flag, IBI HRV metrics, EEG band ratios.
    """
    # Pre-sort all sensor dataframes
    hr_df    = hr_df.sort_values(['pid','timestamp'])
    eda_df   = eda_df.sort_values(['pid','timestamp'])
    temp_df  = temp_df.sort_values(['pid','timestamp'])
    ibi_df   = ibi_df.sort_values(['pid','timestamp'])
    acc_df   = acc_df.sort_values(['pid','timestamp'])
    brain_df = brain_df.sort_values(['pid','timestamp'])

    records = []

    for row_idx, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        # Per-subject baselines for deviation features
        hr_bl_m,   hr_bl_s   = bl_hr.get(pid,   (np.nan, 1))
        eda_bl_m,  eda_bl_s  = bl_eda.get(pid,  (np.nan, 1))
        temp_bl_m, temp_bl_s = bl_temp.get(pid, (np.nan, 1))
        ibi_bl_m,  ibi_bl_s  = bl_ibi.get(pid,  (np.nan, 1))
        acc_bl_m,  acc_bl_s  = bl_acc.get(pid,  (np.nan, 1))

        # ── Add baseline as features ───────────────────────────────
        feat['hr_baseline']   = hr_bl_m
        feat['eda_baseline']  = eda_bl_m
        feat['temp_baseline'] = temp_bl_m
        feat['ibi_baseline']  = ibi_bl_m
        feat['acc_baseline']  = acc_bl_m

        for hw in WINDOWS_MS:
            wlabel = f'w{hw//1000}s'

            # ── HR ────────────────────────────────────────────────
            s_hr = hr_df[hr_df.pid == pid]
            win_hr = s_hr[(s_hr.timestamp >= ts - hw) &
                          (s_hr.timestamp <  ts + hw)]['value'].values
            feat.update(extract_window_stats(win_hr, wlabel, 'hr'))
            if len(win_hr) >= 1:
                feat[f'hr_{wlabel}_dev'] = (np.mean(win_hr) - hr_bl_m) / hr_bl_s
            else:
                feat[f'hr_{wlabel}_dev'] = np.nan

            # ── EDA ───────────────────────────────────────────────
            s_eda = eda_df[eda_df.pid == pid]
            win_eda = s_eda[(s_eda.timestamp >= ts - hw) &
                            (s_eda.timestamp <  ts + hw)]['value'].values
            feat.update(extract_window_stats(win_eda, wlabel, 'eda'))
            if len(win_eda) >= 1:
                zero_ratio = np.mean(win_eda == 0)
                feat[f'eda_{wlabel}_zero_ratio'] = zero_ratio
                feat[f'eda_{wlabel}_valid']       = 1 if zero_ratio < 0.5 else 0
                feat[f'eda_{wlabel}_dev']         = (np.mean(win_eda) - eda_bl_m) / eda_bl_s
            else:
                feat[f'eda_{wlabel}_zero_ratio'] = np.nan
                feat[f'eda_{wlabel}_valid']       = 0
                feat[f'eda_{wlabel}_dev']         = np.nan

            # ── TEMP ──────────────────────────────────────────────
            s_temp = temp_df[temp_df.pid == pid]
            win_temp = s_temp[(s_temp.timestamp >= ts - hw) &
                              (s_temp.timestamp <  ts + hw)]['value'].values
            feat.update(extract_window_stats(win_temp, wlabel, 'temp'))
            if len(win_temp) >= 1:
                feat[f'temp_{wlabel}_dev'] = (np.mean(win_temp) - temp_bl_m) / temp_bl_s
            else:
                feat[f'temp_{wlabel}_dev'] = np.nan

            # ── IBI ───────────────────────────────────────────────
            s_ibi = ibi_df[ibi_df.pid == pid]
            win_ibi = s_ibi[(s_ibi.timestamp >= ts - hw) &
                            (s_ibi.timestamp <  ts + hw)]['value'].values
            if len(win_ibi) >= 2:
                feat[f'ibi_{wlabel}_mean']  = np.mean(win_ibi)
                feat[f'ibi_{wlabel}_std']   = np.std(win_ibi)
                feat[f'ibi_{wlabel}_range'] = np.max(win_ibi) - np.min(win_ibi)
                diffs = np.diff(win_ibi)
                feat[f'ibi_{wlabel}_rmssd'] = np.sqrt(np.mean(diffs**2))
                feat[f'ibi_{wlabel}_dev']   = (np.mean(win_ibi) - ibi_bl_m) / ibi_bl_s
            else:
                for suf in ['mean','std','range','rmssd','dev']:
                    feat[f'ibi_{wlabel}_{suf}'] = np.nan

            # ── ACC ───────────────────────────────────────────────
            s_acc = acc_df[acc_df.pid == pid]
            win_acc = s_acc[(s_acc.timestamp >= ts - hw) &
                            (s_acc.timestamp <  ts + hw)]['magnitude'].values
            if len(win_acc) >= 5:
                feat[f'acc_{wlabel}_mean']   = np.mean(win_acc)
                feat[f'acc_{wlabel}_std']    = np.std(win_acc)
                feat[f'acc_{wlabel}_energy'] = np.mean(win_acc**2)
                feat[f'acc_{wlabel}_dev']    = (np.mean(win_acc) - acc_bl_m) / acc_bl_s
            else:
                for suf in ['mean','std','energy','dev']:
                    feat[f'acc_{wlabel}_{suf}'] = np.nan

            # ── EEG ───────────────────────────────────────────────
            s_eeg = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts - hw) &
                            (s_eeg.timestamp <  ts + hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wlabel}'] = win_eeg[col].mean()
                th = feat.get(f'eeg_theta_{wlabel}', np.nan)
                la = feat.get(f'eeg_lowAlpha_{wlabel}', np.nan)
                ha = feat.get(f'eeg_highAlpha_{wlabel}', np.nan)
                lb = feat.get(f'eeg_lowBeta_{wlabel}', np.nan)
                hb = feat.get(f'eeg_highBeta_{wlabel}', np.nan)
                lg = feat.get(f'eeg_lowGamma_{wlabel}', np.nan)
                feat[f'eeg_theta_alpha_{wlabel}'] = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wlabel}']  = (lb + hb) / (la + ha + eps)
                feat[f'eeg_hbeta_lgamma_{wlabel}']= hb / (lg + eps)
            else:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wlabel}'] = np.nan
                for r in ['theta_alpha','beta_alpha','hbeta_lgamma']:
                    feat[f'eeg_{r}_{wlabel}'] = np.nan

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction function ready.')

Feature extraction function ready.


In [5]:
print('Extracting TRAIN features (3 window sizes — may take 3–5 min)...')
train_feats = extract_all_features(
    train_labels, is_train=True,
    hr_df=trainhr, eda_df=traineda, temp_df=traintemp,
    ibi_df=trainibi, acc_df=trainacc, brain_df=trainbrain,
    bl_hr=bl_tr_hr, bl_eda=bl_tr_eda, bl_temp=bl_tr_temp,
    bl_ibi=bl_tr_ibi, bl_acc=bl_tr_acc
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)

Extracting TRAIN features (3 window sizes — may take 3–5 min)...
Train features: (1456, 165)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, is_train=False,
    hr_df=testhr, eda_df=testeda, temp_df=testtemp,
    ibi_df=testibi, acc_df=testacc, brain_df=testbrain,
    bl_hr=bl_te_hr, bl_eda=bl_te_eda, bl_temp=bl_te_temp,
    bl_ibi=bl_te_ibi, bl_acc=bl_te_acc
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)

Extracting TEST features...
Test features: (1496, 164)


## 3. Temporal Lag Features (previous window context)

In [7]:
# For each subject, the previous label's key features are informative
# because arousal autocorrelation is 0.74-0.94
# We use the ±5s window stats of key sensors from the previous timestamp

LAG_FEATURES = [c for c in train_feats.columns
                if 'w5s' in c and any(s in c for s in ['hr_w5s_mean','eda_w5s_mean',
                                                         'temp_w5s_mean','hr_w5s_dev',
                                                         'eda_w5s_dev'])]


def add_lag_features(feats_df, lag_cols, lag_n=1):
    """Add previous-window (lag) features, within each subject."""
    feats_df = feats_df.sort_values(['pid','timestamp']).copy()
    for col in lag_cols:
        feats_df[f'{col}_lag{lag_n}'] = feats_df.groupby('pid')[col].shift(lag_n)
    return feats_df


train_feats = add_lag_features(train_feats, LAG_FEATURES, lag_n=1)
test_feats  = add_lag_features(test_feats,  LAG_FEATURES, lag_n=1)

# Final feature columns
META_COLS  = ['id', 'pid', 'timestamp', 'arousal']
FEAT_COLS  = [c for c in train_feats.columns if c not in META_COLS]

print(f'Total features after lag: {len(FEAT_COLS)}')
nan_counts = train_feats[FEAT_COLS].isnull().sum()
print('Top NaN columns:')
print(nan_counts[nan_counts > 0].sort_values(ascending=False).head(10))

Total features after lag: 166
Top NaN columns:
ibi_w2s_mean     812
ibi_w2s_std      812
ibi_w2s_range    812
ibi_w2s_rmssd    812
ibi_w2s_dev      812
ibi_w5s_rmssd    631
ibi_w5s_dev      631
ibi_w5s_mean     631
ibi_w5s_std      631
ibi_w5s_range    631
dtype: int64


## 4. LOSO CV — Multi-Seed LightGBM

In [8]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

TRAIN_PIDS = sorted(train_feats['pid'].unique())
N_FOLDS    = len(TRAIN_PIDS)
SEEDS      = [42, 7, 123, 13, 99]   # multi-seed averaging reduces variance

X_all  = train_feats[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats.sort_values(['pid','timestamp'])['arousal'].values - 1).astype(int)
pids   = train_feats.sort_values(['pid','timestamp'])['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

# Re-align after sort
train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values

oof_probs  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_probs = np.zeros((len(test_feats), 5), dtype=np.float64)

# ── LGB params: conservative depth, balanced weights ──────────────────
def get_lgb_params(seed):
    return {
        'objective':        'multiclass',
        'num_class':        5,
        'metric':           'multi_logloss',
        'num_leaves':       31,
        'learning_rate':    0.03,
        'feature_fraction': 0.7,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'min_child_samples':15,
        'lambda_l1':        0.5,
        'lambda_l2':        0.5,
        'max_depth':        6,
        'verbose':         -1,
        'seed':             seed,
        'n_jobs':          -1,
    }


loso_ba_per_fold = []

for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid

    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]

    sw_tr = compute_sample_weight('balanced', y_tr)

    fold_oof_probs  = np.zeros((va_mask.sum(), 5))
    fold_test_probs = np.zeros((len(test_feats), 5))

    for seed in SEEDS:
        params = get_lgb_params(seed)

        dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dval   = lgb.Dataset(X_va, label=y_va, reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(60, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        fold_oof_probs  += model.predict(X_va)    / len(SEEDS)
        fold_test_probs += model.predict(X_test)  / len(SEEDS)

    oof_probs[va_mask]  = fold_oof_probs
    test_probs         += fold_test_probs / N_FOLDS

    pred_va = fold_oof_probs.argmax(axis=1)
    ba = balanced_accuracy_score(y_va, pred_va)
    loso_ba_per_fold.append(ba)
    print(f'  {fold_pid} — BA: {ba:.4f} | n_train={tr_mask.sum()} n_val={va_mask.sum()}')

oof_pred = oof_probs.argmax(axis=1)
oof_ba   = balanced_accuracy_score(y_all, oof_pred)
print(f'\nLOSO mean ± std : {np.mean(loso_ba_per_fold):.4f} ± {np.std(loso_ba_per_fold):.4f}')
print(f'Overall OOF BA  : {oof_ba:.4f}')

  01Z2 — BA: 0.4778 | n_train=1336 n_val=120
  70N8 — BA: 0.1134 | n_train=1312 n_val=144
  7PF3 — BA: 0.1790 | n_train=1309 n_val=147
  CQ2G — BA: 0.2138 | n_train=1335 n_val=121
  D1XP — BA: 0.1893 | n_train=1275 n_val=181
  DT5C — BA: 0.2355 | n_train=1334 n_val=122
  F1ZM — BA: 0.1695 | n_train=1335 n_val=121
  LIUY — BA: 0.1052 | n_train=1330 n_val=126
  SE4Q — BA: 0.5114 | n_train=1334 n_val=122
  TPQI — BA: 0.2500 | n_train=1336 n_val=120
  Y21H — BA: 0.1534 | n_train=1324 n_val=132

LOSO mean ± std : 0.2362 ± 0.1293
Overall OOF BA  : 0.2247


## 5. Diagnosis: Per-Class OOF Performance

In [9]:
# Per-class recall on OOF — shows which arousal levels are being missed
from sklearn.metrics import classification_report
print(classification_report(y_all, oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

# Per-subject OOF BA
print('\nPer-subject OOF BA:')
for pid, ba in zip(TRAIN_PIDS, loso_ba_per_fold):
    flag = ' ← LOW' if ba < 0.20 else ''
    print(f'  {pid}: {ba:.4f}{flag}')

              precision    recall  f1-score   support

   Arousal 1       0.02      0.05      0.03        55
   Arousal 2       0.30      0.27      0.28       430
   Arousal 3       0.44      0.34      0.38       554
   Arousal 4       0.35      0.46      0.40       345
   Arousal 5       0.00      0.00      0.00        72

    accuracy                           0.32      1456
   macro avg       0.22      0.22      0.22      1456
weighted avg       0.34      0.32      0.32      1456


Per-subject OOF BA:
  01Z2: 0.4778
  70N8: 0.1134 ← LOW
  7PF3: 0.1790 ← LOW
  CQ2G: 0.2138
  D1XP: 0.1893 ← LOW
  DT5C: 0.2355
  F1ZM: 0.1695 ← LOW
  LIUY: 0.1052 ← LOW
  SE4Q: 0.5114
  TPQI: 0.2500
  Y21H: 0.1534 ← LOW


## 6. Generate Final Test Predictions

In [10]:
# Final test predictions
test_pred_labels = test_probs.argmax(axis=1) + 1  # back to 1–5

print('Test prediction distribution:')
print(pd.Series(test_pred_labels).value_counts().sort_index())

# Sanity: test should have more variety than {3: everything}
n_unique = len(np.unique(test_pred_labels))
print(f'Unique predicted classes: {n_unique}/5')

Test prediction distribution:
1    176
2    250
3    649
4    303
5    118
Name: count, dtype: int64
Unique predicted classes: 5/5


# Generating Final Submissions


In [11]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred_labels,
})

submission.to_csv('submission_v13.csv', index=False)
print('submission.csv saved.')
print(f'Shape: {submission.shape}')
print(submission.head(10))

submission.csv saved.
Shape: (1496, 2)
     id  arousal
0  2054        3
1  1456        3
2  2055        3
3  1457        3
4  1458        3
5  2056        3
6  2057        3
7  2058        3
8  2059        3
9  2060        3
